# QED-C Application-Oriented Benchmarks - Modularized with qBraid Execution

This notebook runs all supported benchmarks using the modularized approach with qBraid/Equal1 execution backend.

**Key Features:**
- Modularized problem generation using `get_circuits=True`
- Execution via qBraid API with Equal1 backend
- Configurable global parameters for all benchmarks
- Automatic metrics collection and visualization

## Global Configuration

Configure all benchmark parameters in one place.

In [1]:
# Benchmark execution parameters
min_qubits = 6
max_qubits = 6
skip_qubits = 1
max_circuits = 2
num_shots = 1000

# qBraid/Equal1 device configuration
equal1_device = "equal1_simulator"
equal1_noise_model = "bell2-17-gen-preview"
simulation_backend = "DensityMatrix"  # Options: "DensityMatrix", "StateVector", etc.
simulation_platform = "CPU"  # Options: "CPU", "GPU"
optimization_level = 2

# Benchmark specific overrides (optional)
vqe_num_shots = 4098

## Setup qBraid Execution Environment

Initialize the qBraid executor that will be used for all benchmarks.

In [2]:
import base64
from qbraid import QbraidProvider
from qiskit import QuantumCircuit, qasm2, qasm3


class QBraidBackEnd():
    def __init__(self):
        self.name = "QBraidEqual1Backend"


class QBraidResult:
    def __init__(self, counts, exec_time, transpiled_circuit_metrics):
        self.exec_time = exec_time
        self.counts = counts
        self.transpiled_circuit_metrics = transpiled_circuit_metrics

    def get_counts(self, qc):
        return self.counts

    def get_transpiled_circuit_metrics(self):
        return self.transpiled_circuit_metrics


class QBraidExecutor():
    def __init__(self, device_name, noise_model, backend, platform, opt_level):
        provider = QbraidProvider()
        self.device = provider.get_device(device_name)
        self.noise_model = noise_model
        self.backend = backend
        self.platform = platform
        self.opt_level = opt_level

    def __call__(self, qc: QuantumCircuit, backend_name: str, backend, shots, **kwargs) -> QBraidResult:
        print(f"Running circuit on {backend_name} ({self.backend}/{self.platform}) with {shots} shots")

        runtime_options = {
            "simulation_platform": self.platform,
            "execution_options": {"optimization_level": self.opt_level},
        }

        job = self.device.run(
            qc,
            shots=shots,
            noise_model=self.noise_model,
            runtime_options=runtime_options,
            backend=self.backend
        )
        job.wait_for_final_state()

        if job.status().name != "COMPLETED":
            job_result = job.client.get_job_results(job.id)
            print(f"\n{'='*80}\nJOB FAILED\nError: {job_result['statusText']}\n{'='*80}\n")
            raise RuntimeError(f"Job failed: {job_result['statusText']}")

        result = job.result()
        counts = result.data.get_counts()

        result_json = job.client.get_job_results(job.id)
        inner_exec_time = result_json['executionMetrics']['executor']

        transpiled_circuit = base64.b64decode(result_json['compiledOutput']).decode('utf-8')
        try:
            transpiled_qc = qasm2.loads(transpiled_circuit, custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS)
        except qasm2.exceptions.QASM2ParseError:
            try:
                transpiled_qc = qasm3.loads(transpiled_circuit)
            except qasm3.exceptions.QASM3Error as e:
                print(f"Warning: Could not parse transpiled circuit: {e}")
                transpiled_qc = None

        from _common.qiskit.execute import get_circuit_metrics
        metrics = get_circuit_metrics(transpiled_qc) if transpiled_qc else {}

        return QBraidResult(counts, inner_exec_time, metrics)


# Initialize the executor with global configuration
executor = QBraidExecutor(
    equal1_device,
    equal1_noise_model,
    simulation_backend,
    simulation_platform,
    optimization_level
)

# Backend configuration for benchmarks
backend_id = "qasm_simulator"
exec_options = {"executor": executor}

print(f"✓ Executor initialized: {equal1_device}")
print(f"  Backend: {simulation_backend}")
print(f"  Platform: {simulation_platform}")
print(f"  Noise Model: {equal1_noise_model}")

/home/iszilveszter/work/cuda-quantum/QC-App-Oriented-Benchmarks/.venv/lib64/python3.12/site-packages/qbraid_core/_compat.py:44: UserWarning: You are using qbraid-core version 0.1.41, however, version 0.2.0 is available. To avoid compatibility issues, consider upgrading.
  warnings.warn(


✓ Executor initialized: equal1_simulator
  Backend: DensityMatrix
  Platform: CPU
  Noise Model: bell2-17-gen-preview


/home/iszilveszter/work/cuda-quantum/QC-App-Oriented-Benchmarks/.venv/lib64/python3.12/site-packages/qbraid/runtime/native/provider.py:151: RuntimeWarning: The default runtime configuration for device 'equal1_simulator' includes transpilation to program type 'cudaq', which is not registered.
  warnings.warn(


## Helper Functions for Modularized Execution

These functions handle the common workflow for all benchmarks.

In [3]:
from _common import metrics
from _common.qiskit import execute as ex


def prepare_circuits_and_metrics(circuits, metadata):
    """
    Convert circuits dictionary to flat list and prepare metrics.

    Returns:
        tuple: (circuit_identifiers, flat_circuits)
    """
    metadata.pop("subtitle", None)
    metrics.circuit_metrics = metadata.copy()

    circuit_identifiers = []
    flat_circuits = []

    for num_qubits in circuits.keys():
        for circuit_id in circuits[num_qubits].keys():
            circuit_identifiers.append((num_qubits, circuit_id))
            flat_circuits.append(circuits[num_qubits][circuit_id])

            ex.compute_and_store_circuit_info(
                circuits[num_qubits][circuit_id],
                str(num_qubits),
                str(circuit_id),
                do_transpile_metrics=True,
                use_normalized_depth=True,
            )

    return circuit_identifiers, flat_circuits


def execute_circuits(flat_circuits, shots):
    """
    Execute circuits using qBraid executor.

    Returns:
        list: Results for each circuit
    """
    results = []
    for idx, qc in enumerate(flat_circuits):
        print(f"  Executing circuit {idx+1}/{len(flat_circuits)}...")
        result = executor(qc, backend_id, None, shots)
        results.append(result)
    return results


def analyze_and_plot(benchmark_name, circuit_identifiers, results, analyze_func, num_shots, **analyze_kwargs):
    """
    Analyze results and create plots.
    """
    for idx, (num_qubits, circuit_id) in enumerate(circuit_identifiers):
        counts = results[idx].get_counts(None)
        _, fidelity = analyze_func(
            None, results[idx], int(num_qubits), num_shots,
            **analyze_kwargs
        )
        metrics.store_metric(num_qubits, circuit_id, "fidelity", fidelity)

    metrics.aggregate_metrics()
    subtitle = f"Benchmark Results - {benchmark_name}"
    metrics.circuit_metrics["subtitle"] = f"device = {equal1_device} ({simulation_backend}/{simulation_platform})"

    filters = ["fidelity", "hf_fidelity", "depth", "2q", "vbplot"]
    metrics.plot_metrics(subtitle, filters=filters)


print("✓ Helper functions loaded")

✓ Helper functions loaded


## 1. Deutsch-Jozsa

In [5]:
print("="*80)
print("Running Deutsch-Jozsa Benchmark")
print("="*80)

from deutsch_jozsa import dj_benchmark

circuits, metadata = dj_benchmark.run(
    min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
    max_circuits=max_circuits, num_shots=num_shots, get_circuits=True
)

circuit_identifiers, flat_circuits = prepare_circuits_and_metrics(circuits, metadata)
print(f"Generated {len(flat_circuits)} circuits")

results = execute_circuits(flat_circuits, num_shots)

analyze_and_plot(
    "Deutsch-Jozsa",
    circuit_identifiers,
    results,
    dj_benchmark.analyze_and_print_result,
    num_shots
)

print("✓ Deutsch-Jozsa completed\n")

Running Deutsch-Jozsa Benchmark
Deutsch-Jozsa Benchmark Program - Qiskit
... execution starting at Feb 25, 2026 14:36:43 UTC
************
Creating [2] circuits with num_qubits = 6
************
Returning circuits and circuit information
Generated 2 circuits
  Executing circuit 1/2...
Running circuit on qasm_simulator (DensityMatrix/CPU) with 1000 shots


Traceback (most recent call last):
  File "_pydevd_bundle\\pydevd_cython.pyx", line 1609, in _pydevd_bundle.pydevd_cython.handle_exception
  File "/home/iszilveszter/work/cuda-quantum/QC-App-Oriented-Benchmarks/.venv/lib64/python3.12/site-packages/debugpy/_vendored/pydevd/pydevd.py", line 2188, in do_wait_suspend
    keep_suspended = self._do_wait_suspend(thread, frame, event, arg, trace_suspend_type, from_this_thread, frames_tracker)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/iszilveszter/work/cuda-quantum/QC-App-Oriented-Benchmarks/.venv/lib64/python3.12/site-packages/debugpy/_vendored/pydevd/pydevd.py", line 2257, in _do_wait_suspend
    notify_event.wait(wait_timeout)
  File "/usr/lib64/python3.12/threading.py", line 655, in wait
    signaled = self._cond.wait(timeout)
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib64/python3.12/threading.py", line 359, in wait
    gotit = waiter

QASM3ParsingError: 

## 2. Bernstein-Vazirani (Method 1)

In [ ]:
print("="*80)
print("Running Bernstein-Vazirani (Method 1) Benchmark")
print("="*80)

from bernstein_vazirani import bv_benchmark

circuits, metadata = bv_benchmark.run(
    min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
    max_circuits=max_circuits, num_shots=num_shots, method=1, get_circuits=True
)

circuit_identifiers, flat_circuits = prepare_circuits_and_metrics(circuits, metadata)
print(f"Generated {len(flat_circuits)} circuits")

results = execute_circuits(flat_circuits, num_shots)

for idx, (num_qubits, circuit_id) in enumerate(circuit_identifiers):
    counts = results[idx].get_counts(None)
    _, fidelity = bv_benchmark.analyze_and_print_result(
        None, results[idx], int(num_qubits), num_shots, s_int=int(circuit_id)
    )
    metrics.store_metric(num_qubits, circuit_id, "fidelity", fidelity)

metrics.aggregate_metrics()
subtitle = f"Benchmark Results - Bernstein-Vazirani (Method 1)"
metrics.circuit_metrics["subtitle"] = f"device = {equal1_device} ({simulation_backend}/{simulation_platform})"
metrics.plot_metrics(subtitle, filters=["fidelity", "hf_fidelity", "depth", "2q", "vbplot"])

print("✓ Bernstein-Vazirani completed\n")

## 3. Hidden Shift

In [ ]:
print("="*80)
print("Running Hidden Shift Benchmark")
print("="*80)

from hidden_shift import hs_benchmark

circuits, metadata = hs_benchmark.run(
    min_qubits=min_qubits, max_qubits=max_qubits,
    max_circuits=max_circuits, num_shots=num_shots, get_circuits=True
)

circuit_identifiers, flat_circuits = prepare_circuits_and_metrics(circuits, metadata)
print(f"Generated {len(flat_circuits)} circuits")

results = execute_circuits(flat_circuits, num_shots)

analyze_and_plot(
    "Hidden Shift",
    circuit_identifiers,
    results,
    hs_benchmark.analyze_and_print_result,
    num_shots
)

print("✓ Hidden Shift completed\n")

## 4. Quantum Fourier Transform (Method 1)

In [ ]:
print("="*80)
print("Running Quantum Fourier Transform (Method 1) Benchmark")
print("="*80)

from quantum_fourier_transform import qft_benchmark

circuits, metadata = qft_benchmark.run(
    min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
    max_circuits=max_circuits, num_shots=num_shots, method=1, get_circuits=True
)

circuit_identifiers, flat_circuits = prepare_circuits_and_metrics(circuits, metadata)
print(f"Generated {len(flat_circuits)} circuits")

results = execute_circuits(flat_circuits, num_shots)

analyze_and_plot(
    "Quantum Fourier Transform (Method 1)",
    circuit_identifiers,
    results,
    qft_benchmark.analyze_and_print_result,
    num_shots
)

print("✓ QFT Method 1 completed\n")

## 5. Quantum Fourier Transform (Method 2)

In [ ]:
print("="*80)
print("Running Quantum Fourier Transform (Method 2) Benchmark")
print("="*80)

from quantum_fourier_transform import qft_benchmark

circuits, metadata = qft_benchmark.run(
    min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
    max_circuits=max_circuits, num_shots=num_shots, method=2, get_circuits=True
)

circuit_identifiers, flat_circuits = prepare_circuits_and_metrics(circuits, metadata)
print(f"Generated {len(flat_circuits)} circuits")

results = execute_circuits(flat_circuits, num_shots)

analyze_and_plot(
    "Quantum Fourier Transform (Method 2)",
    circuit_identifiers,
    results,
    qft_benchmark.analyze_and_print_result,
    num_shots
)

print("✓ QFT Method 2 completed\n")

## 6. Grover's Search

In [ ]:
print("="*80)
print("Running Grover's Search Benchmark")
print("="*80)

from grovers import grovers_benchmark

circuits, metadata = grovers_benchmark.run(
    min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
    max_circuits=max_circuits, num_shots=num_shots, get_circuits=True
)

circuit_identifiers, flat_circuits = prepare_circuits_and_metrics(circuits, metadata)
print(f"Generated {len(flat_circuits)} circuits")

results = execute_circuits(flat_circuits, num_shots)

analyze_and_plot(
    "Grover's Search",
    circuit_identifiers,
    results,
    grovers_benchmark.analyze_and_print_result,
    num_shots
)

print("✓ Grover's Search completed\n")

## 7. Phase Estimation

In [ ]:
print("="*80)
print("Running Phase Estimation Benchmark")
print("="*80)

from phase_estimation import pe_benchmark

circuits, metadata = pe_benchmark.run(
    min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
    max_circuits=max_circuits, num_shots=num_shots, get_circuits=True
)

circuit_identifiers, flat_circuits = prepare_circuits_and_metrics(circuits, metadata)
print(f"Generated {len(flat_circuits)} circuits")

results = execute_circuits(flat_circuits, num_shots)

for idx, (num_qubits, circuit_id) in enumerate(circuit_identifiers):
    counts = results[idx].get_counts(None)
    _, fidelity = pe_benchmark.analyze_and_print_result(
        None, results[idx], int(num_qubits), float(circuit_id), num_shots
    )
    metrics.store_metric(num_qubits, circuit_id, "fidelity", fidelity)

metrics.aggregate_metrics()
subtitle = f"Benchmark Results - Phase Estimation"
metrics.circuit_metrics["subtitle"] = f"device = {equal1_device} ({simulation_backend}/{simulation_platform})"
metrics.plot_metrics(subtitle, filters=["fidelity", "hf_fidelity", "depth", "2q", "vbplot"])

print("✓ Phase Estimation completed\n")

## 8. HHL Linear Solver

In [ ]:
print("="*80)
print("Running HHL Linear Solver Benchmark")
print("="*80)

from hhl.qiskit import hhl_benchmark

hhl_benchmark.verbose = False

circuits, metadata = hhl_benchmark.run(
    min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
    max_circuits=max_circuits, num_shots=num_shots,
    method=1, use_best_widths=True, get_circuits=True
)

circuit_identifiers, flat_circuits = prepare_circuits_and_metrics(circuits, metadata)
print(f"Generated {len(flat_circuits)} circuits")

results = execute_circuits(flat_circuits, num_shots)

analyze_and_plot(
    "HHL Linear Solver",
    circuit_identifiers,
    results,
    hhl_benchmark.analyze_and_print_result,
    num_shots
)

print("✓ HHL completed\n")

## 9. Amplitude Estimation

In [ ]:
print("="*80)
print("Running Amplitude Estimation Benchmark")
print("="*80)

from amplitude_estimation import ae_benchmark

circuits, metadata = ae_benchmark.run(
    min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
    max_circuits=max_circuits, num_shots=num_shots, get_circuits=True
)

circuit_identifiers, flat_circuits = prepare_circuits_and_metrics(circuits, metadata)
print(f"Generated {len(flat_circuits)} circuits")

results = execute_circuits(flat_circuits, num_shots)

analyze_and_plot(
    "Amplitude Estimation",
    circuit_identifiers,
    results,
    ae_benchmark.analyze_and_print_result,
    num_shots
)

print("✓ Amplitude Estimation completed\n")

## 10. Monte Carlo

In [ ]:
print("="*80)
print("Running Monte Carlo Benchmark")
print("="*80)

from monte_carlo import mc_benchmark

circuits, metadata = mc_benchmark.run(
    min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
    max_circuits=max_circuits, num_shots=num_shots, get_circuits=True
)

circuit_identifiers, flat_circuits = prepare_circuits_and_metrics(circuits, metadata)
print(f"Generated {len(flat_circuits)} circuits")

results = execute_circuits(flat_circuits, num_shots)

analyze_and_plot(
    "Monte Carlo",
    circuit_identifiers,
    results,
    mc_benchmark.analyze_and_print_result,
    num_shots
)

print("✓ Monte Carlo completed\n")

## 11. Hamiltonian Simulation (Method 1)

In [ ]:
print("="*80)
print("Running Hamiltonian Simulation (Method 1) Benchmark")
print("="*80)

from hamiltonian_simulation import hamiltonian_simulation_benchmark

circuits, metadata = hamiltonian_simulation_benchmark.run(
    min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
    max_circuits=max_circuits, num_shots=num_shots, method=1, get_circuits=True
)

circuit_identifiers, flat_circuits = prepare_circuits_and_metrics(circuits, metadata)
print(f"Generated {len(flat_circuits)} circuits")

results = execute_circuits(flat_circuits, num_shots)

analyze_and_plot(
    "Hamiltonian Simulation (Method 1)",
    circuit_identifiers,
    results,
    hamiltonian_simulation_benchmark.analyze_and_print_result,
    num_shots
)

print("✓ Hamiltonian Simulation completed\n")

## 12. VQE (Method 1)

In [ ]:
print("="*80)
print("Running VQE (Method 1) Benchmark")
print("="*80)

from vqe.qiskit import vqe_benchmark

circuits, metadata = vqe_benchmark.run(
    min_qubits=min_qubits, max_qubits=max_qubits,
    max_circuits=max_circuits, num_shots=vqe_num_shots, method=1, get_circuits=True
)

circuit_identifiers, flat_circuits = prepare_circuits_and_metrics(circuits, metadata)
print(f"Generated {len(flat_circuits)} circuits")

results = execute_circuits(flat_circuits, vqe_num_shots)

analyze_and_plot(
    "VQE (Method 1)",
    circuit_identifiers,
    results,
    vqe_benchmark.analyze_and_print_result,
    vqe_num_shots
)

print("✓ VQE completed\n")

## 13. Shor's Algorithm (Method 1)

In [ ]:
print("="*80)
print("Running Shor's Algorithm (Method 1) Benchmark")
print("="*80)

from shors.qiskit import shors_benchmark

circuits, metadata = shors_benchmark.run(
    min_qubits=min_qubits, max_qubits=max_qubits,
    max_circuits=1, num_shots=num_shots, method=1, get_circuits=True
)

circuit_identifiers, flat_circuits = prepare_circuits_and_metrics(circuits, metadata)
print(f"Generated {len(flat_circuits)} circuits")

results = execute_circuits(flat_circuits, num_shots)

analyze_and_plot(
    "Shor's Algorithm (Method 1)",
    circuit_identifiers,
    results,
    shors_benchmark.analyze_and_print_result,
    num_shots
)

print("✓ Shor's Method 1 completed\n")

## 14. Shor's Algorithm (Method 2)

In [ ]:
print("="*80)
print("Running Shor's Algorithm (Method 2) Benchmark")
print("="*80)

from shors.qiskit import shors_benchmark

circuits, metadata = shors_benchmark.run(
    min_qubits=min_qubits, max_qubits=max_qubits,
    max_circuits=1, num_shots=num_shots, method=2, get_circuits=True
)

circuit_identifiers, flat_circuits = prepare_circuits_and_metrics(circuits, metadata)
print(f"Generated {len(flat_circuits)} circuits")

results = execute_circuits(flat_circuits, num_shots)

analyze_and_plot(
    "Shor's Algorithm (Method 2)",
    circuit_identifiers,
    results,
    shors_benchmark.analyze_and_print_result,
    num_shots
)

print("✓ Shor's Method 2 completed\n")

## Combined Results Summary

View aggregate metrics across all benchmarks.

In [ ]:
import sys
sys.path.insert(1, "_common")
import metrics

metrics.plot_all_app_metrics(backend_id, do_all_plots=False, include_apps=None)

print("\n" + "="*80)
print("ALL BENCHMARKS COMPLETED")
print("="*80)

## Utility: Close Session

Execute this cell if you need to manually close an active session after abnormal termination.

In [ ]:
import sys
sys.path.insert(1, "_common")
import execute as ex

ex.close_session()
print("✓ Session closed")